In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from src.core.data_splitter import DataSpliter
from src.core.data_preprocessing import (
    assign_class, PipelineA, PipelineB
)

In [2]:
RAW_PATH    = Path('../data/raw/urban_pluvial_flood_risk_dataset.xlsx')
SPLITS_DIR  = Path('../data/splits')
PROCESSED_DIR = Path('../data/processed')

In [3]:
df = pd.read_excel(RAW_PATH)
print(f"Raw data: {df.shape}")

# Tạo risk_class TRƯỚC khi split
# (cần để stratify theo class, không phải raw risk_labels)
df['risk_class'] = df['risk_labels'].apply(assign_class)
print(f"\nClass distribution:")
print(df['risk_class'].value_counts())
print(df['risk_class'].value_counts(normalize=True).round(3))

Raw data: (2963, 17)

Class distribution:
risk_class
Low       1994
Medium     574
High       395
Name: count, dtype: int64
risk_class
Low       0.673
Medium    0.194
High      0.133
Name: proportion, dtype: float64


In [4]:
splitter = DataSpliter(
    target_col   = 'risk_class',
    test_size    = 0.20,
    random_state = 42
)
splitter.split(df)
train_df, test_df = splitter.get_splits()

# Lưu raw splits
splitter.save(SPLITS_DIR)


Total samples: 2963
Train samples: 2370 (79.99%)
Test samples: 593 (20.01%)


In [5]:
import importlib
import src.core.data_preprocessing as dp

importlib.reload(dp)


<module 'src.core.data_preprocessing' from 'c:\\Users\\LENOVO\\Downloads\\khu lap trinh\\machine_learning\\flood-forecast\\experiments\\..\\src\\core\\data_preprocessing.py'>

In [6]:
print("\n" + "="*60)
print("TREE PREPROCESSOR")
print("="*60)
tree_prep  = PipelineA()
train_tree = tree_prep.fit_transform(train_df)
test_tree  = tree_prep.transform(test_df)

print("\n" + "="*60)
print("LINEAR PREPROCESSOR")
print("="*60)
linear_prep  = PipelineB()
train_linear = linear_prep.fit_transform(train_df)
test_linear  = linear_prep.transform(test_df)


TREE PREPROCESSOR
[PipelineA] fitting on 2,370 rows, fe_groups=['G1', 'G2', 'G3', 'G4']...
[PipelineA] done — 14 features

LINEAR PREPROCESSOR

[PipelineB] fitting on 2,370 rows...
  use_skew=True, use_scale=True, use_ohe=True, fe_groups=['G1', 'G2', 'G3', 'G4']
  Yeo-Johnson transforms:
    'elevation_m': skew=1.788 → λ=0.4047
    'storm_drain_proximity_m': skew=1.126 → λ=0.3486
    'historical_rainfall_intensity_mm_hr': skew=1.438 → λ=-0.0025
    'return_period_years': skew=1.982 → λ=-0.2557
    'G1_infra_vuln': skew=2.956 → λ=0.0766
    'G2_rain_x_return': skew=3.598 → λ=-0.1113
    'G3_soil_x_rainfall': skew=1.802 → λ=0.4177
[PipelineB] done — 34 features


In [7]:
print("\n" + "="*60)
print("KIỂM TRA OUTPUT")
print("="*60)

for name, df_out in [
    ('train_tree'  , train_tree),
    ('test_tree'   , test_tree),
    ('train_linear', train_linear),
    ('test_linear' , test_linear),
]:
    print(f"\n{name}:")
    print(f"  Shape  : {df_out.shape}")
    print(f"  Columns: {list(df_out.columns)}")
    print(f"  NaN    : {df_out.isnull().sum().sum()}")
    print(f"  Classes: {df_out['risk_class'].value_counts().to_dict()}")


KIỂM TRA OUTPUT

train_tree:
  Shape  : (2370, 18)
  Columns: ['elevation_m', 'drainage_density_km_per_km2', 'storm_drain_proximity_m', 'historical_rainfall_intensity_mm_hr', 'return_period_years', 'G1_infra_vuln', 'G2_rain_x_return', 'G3_soil_x_rainfall', 'G4_is_very_low_elev', 'land_use', 'soil_group', 'storm_drain_type', 'rainfall_source', 'dem_source', 'risk_class', 'risk_class_encoded', 'latitude', 'longitude']
  NaN    : 0
  Classes: {'Low': 1595, 'Medium': 459, 'High': 316}

test_tree:
  Shape  : (593, 18)
  Columns: ['elevation_m', 'drainage_density_km_per_km2', 'storm_drain_proximity_m', 'historical_rainfall_intensity_mm_hr', 'return_period_years', 'G1_infra_vuln', 'G2_rain_x_return', 'G3_soil_x_rainfall', 'G4_is_very_low_elev', 'land_use', 'soil_group', 'storm_drain_type', 'rainfall_source', 'dem_source', 'risk_class', 'risk_class_encoded', 'latitude', 'longitude']
  NaN    : 0
  Classes: {'Low': 399, 'Medium': 115, 'High': 79}

train_linear:
  Shape  : (2370, 38)
  Columns:

In [8]:
def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"  Saved: {path} ({len(df):,} rows x {df.shape[1]} cols)")

print("\nSaving processed data...")
save_csv(train_tree,   PROCESSED_DIR / 'tree/train.csv')
save_csv(test_tree,    PROCESSED_DIR / 'tree/test.csv')
save_csv(train_linear, PROCESSED_DIR / 'linear/train.csv')
save_csv(test_linear,  PROCESSED_DIR / 'linear/test.csv')

print("\nDone! Cấu trúc:")
for p in sorted(PROCESSED_DIR.rglob('*.csv')):
    print(f"  {p}")


Saving processed data...
  Saved: ..\data\processed\tree\train.csv (2,370 rows x 18 cols)
  Saved: ..\data\processed\tree\test.csv (593 rows x 18 cols)
  Saved: ..\data\processed\linear\train.csv (2,370 rows x 38 cols)
  Saved: ..\data\processed\linear\test.csv (593 rows x 38 cols)

Done! Cấu trúc:
  ..\data\processed\linear\test.csv
  ..\data\processed\linear\train.csv
  ..\data\processed\tree\test.csv
  ..\data\processed\tree\train.csv
